# Week 2 — Dataset Exploration and Data Quality Assessment
**Project:** Municipal Service Delivery Analytics — Eight-Week Capstone  
**Milestone:** Week 2 only (no cleaning or modeling performed)  
**Source:** `data/raw/Municipal_Service_Data.csv` (provided dataset; city, provenance and license unspecified).

**Goal:** Inspect the raw dataset's structure, completeness, unique identifiers, categorical distributions, date consistency, and limitations. All outputs below were executed against the included original CSV.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Works when launched from either repository root or notebooks/ folder.
root = Path.cwd()
if not (root / 'data' / 'raw' / 'Municipal_Service_Data.csv').exists():
    root = root.parent
path = root / 'data' / 'raw' / 'Municipal_Service_Data.csv'
assert path.exists(), f"Dataset not found: {path}"
df = pd.read_csv(path)
print('Dataset shape (rows, columns):', df.shape)
display(df.head())

Dataset shape (rows, columns): (2000, 11)


,Service_Request_ID,Request_Type,Description,Department,Priority_Level,Status,Date_Opened,Date_Closed,Ward_District,Latitude,Longitude
0,SR00001,Garbage Collection,Service request related to municipal operations.,Public Works,High,Closed,2022-04-13 19:00:00,2022-04-28 08:00:00,South,49.233907,-123.020945
1,SR00002,Water Leakage,Service request related to municipal operations.,Public Works,Low,Open,2022-05-02 18:00:00,NaN,West,49.259853,-123.019665
2,SR00003,Pothole Repair,Service request related to municipal operations.,Public Works,Low,Closed,2023-10-26 02:00:00,NaN,South,49.216600,-122.918434
3,SR00004,Sewer Issue,Service request related to municipal operations.,Parks & Recreation,Low,Open,2023-01-21 20:00:00,2023-01-28 13:00:00,South,49.157427,-123.070416
4,SR00005,Sewer Issue,Service request related to municipal operations.,Electrical Services,Medium,In Progress,2023-07-15 09:00:00,2023-07-23 05:00:00,Central,49.136821,-122.964447


## 1. Structure, types and descriptive overview
Date fields are initially strings; conversion is inspected separately without changing the source CSV.

In [2]:
schema = pd.DataFrame({'dtype': df.dtypes.astype(str), 'non_null': df.notna().sum(), 'unique_non_null': df.nunique(dropna=True)})
display(schema)
display(df.describe(include='all').T)

,dtype,non_null,unique_non_null
Service_Request_ID,object,2000,2000
Request_Type,object,2000,9
Description,object,2000,1
Department,object,2000,5
Priority_Level,object,2000,4
Status,object,2000,3
Date_Opened,object,2000,1901
Date_Closed,object,1815,1732
Ward_District,object,2000,5
Latitude,float64,2000,1993


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Service_Request_ID,2000,2000,SR00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Request_Type,2000,9,Noise Complaint,245,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,2000,1,Service request related to municipal operations.,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Department,2000,5,Electrical Services,420,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Priority_Level,2000,4,Medium,524,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,2000,3,Closed,1198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date_Opened,2000,1901,2023-07-14 18:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date_Closed,1815,1732,2023-08-20 22:00:00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ward_District,2000,5,South,422,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,2000.0,NaN,NaN,NaN,49.150212,0.085476,49.000071,49.078252,49.148889,49.221441,49.299852


## 2. Completeness and uniqueness

In [3]:
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)})
display(missing)
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Duplicate Service_Request_ID values:', int(df['Service_Request_ID'].duplicated().sum()))
print('Unique request IDs:', df['Service_Request_ID'].nunique())

,missing_count,missing_pct
Service_Request_ID,0,0.00
Request_Type,0,0.00
Description,0,0.00
Department,0,0.00
Priority_Level,0,0.00
Status,0,0.00
Date_Opened,0,0.00
Date_Closed,185,9.25
Ward_District,0,0.00
Latitude,0,0.00


Exact duplicate rows: 0
Duplicate Service_Request_ID values: 0
Unique request IDs: 2000


## 3. Categorical distributions

In [4]:
for col in ['Request_Type', 'Department', 'Priority_Level', 'Status', 'Ward_District']:
    print(f'\n{col} (including missing values):')
    display(df[col].value_counts(dropna=False).rename_axis(col).to_frame('requests'))


Request_Type (including missing values):


,requests
Request_Type,
Noise Complaint,245
Road Damage,232
Water Leakage,228
Tree Maintenance,227
Sewer Issue,221
Pothole Repair,219
Street Light Outage,214
Garbage Collection,208
Graffiti Removal,206



Department (including missing values):


,requests
Department,
Electrical Services,420
Water Services,402
Parks & Recreation,401
Public Works,389
Sanitation,388



Priority_Level (including missing values):


,requests
Priority_Level,
Medium,524
Emergency,502
Low,498
High,476



Status (including missing values):


,requests
Status,
Closed,1198
In Progress,419
Open,383



Ward_District (including missing values):


,requests
Ward_District,
South,422
West,422
North,394
East,384
Central,378


## 4. Timestamp validity and status/date inconsistencies
These checks **flag** possible issues, not silently repair them. A populated closure date does not establish that a request is marked Closed.

In [5]:
opened = pd.to_datetime(df['Date_Opened'], errors='coerce')
closed = pd.to_datetime(df['Date_Closed'], errors='coerce')
print('Unparseable non-null opening timestamps:', int((df['Date_Opened'].notna() & opened.isna()).sum()))
print('Unparseable non-null closing timestamps:', int((df['Date_Closed'].notna() & closed.isna()).sum()))
print('Opening date range:', opened.min(), 'to', opened.max())
print('Closing date range:', closed.min(), 'to', closed.max())
closed_status = df['Status'].astype(str).str.strip().eq('Closed')
no_close_time = closed_status & closed.isna()
nonclosed_with_time = ~closed_status & closed.notna()
print('Closed status with missing closure timestamp:', int(no_close_time.sum()))
print('Non-Closed status with closure timestamp:', int(nonclosed_with_time.sum()))
print('Total status/date mismatches:', int((no_close_time | nonclosed_with_time).sum()))
print('Closure before opening:', int((closed.notna() & opened.notna() & (closed < opened)).sum()))
print('Missing latitude/longitude:', int(df[['Latitude', 'Longitude']].isna().any(axis=1).sum()))

Unparseable non-null opening timestamps: 0
Unparseable non-null closing timestamps: 0
Opening date range: 2022-01-01 13:00:00 to 2023-12-31 15:00:00
Closing date range: 2022-01-04 04:00:00 to 2024-01-14 10:00:00
Closed status with missing closure timestamp: 98
Non-Closed status with closure timestamp: 715
Total status/date mismatches: 813
Closure before opening: 0
Missing latitude/longitude: 0


## 5. Week 2 observations and interpretation

- **Size:** 2,000 records and 11 original variables; request IDs are unique in the supplied CSV.
- **Missingness:** 185 `Date_Closed` values are missing (9.25%). Missing closure dates require status-aware interpretation, not automatic deletion.
- **Data-quality warning:** 98 records marked `Closed` have no closure timestamp, while 715 records not marked `Closed` have a closure timestamp (813 total mismatches). These are flags for Week 3 review; do not equate a closure timestamp with a valid closed case.
- **Categorical structure:** Nine request types, five departments, four priority levels, three statuses and five districts. The `Description` field has one repeated value, so it carries little analytical detail.
- **Date fields:** Loaded as text in the raw CSV; datetime parsing and consistency checks are necessary before resolution-time calculations.
- **Scope limitations:** First-response timestamps, service-level targets and staffing data are absent. Therefore response-time compliance and causal explanations for delays cannot be established. Original municipality and licensing are unspecified.

**Week 3 handoff:** Investigate date/status conflicts, document any cleaning rules, convert valid date fields, and retain the original dataset unchanged. No cleaning has been performed in this Week 2 submission.